In [ ]:
import os 
import pandas as pd

In [ ]:
prev_dir = "./data/PseudoPFLOW/ver2.0/activity/8/"
processed_dir = "./data/activity_merge/8/"
output_dir = "./data/PseudoPFLOW/ver2.0/activity/8/updated/"

os.makedirs(output_dir, exist_ok=True)

for file_name in os.listdir(prev_dir):
    if file_name.endswith(".csv"):
        prev_file_path = os.path.join(prev_dir, file_name)
        processed_file_path = os.path.join(processed_dir, file_name)
        
        # Check if the corresponding file exists in processed_dir
        if not os.path.exists(processed_file_path):
            print(f"File {processed_file_path} does not exist. Skipping...")
            continue

        act_prev = pd.read_csv(prev_file_path, names=['pid', 'age', 'gender', 'occupation', 'starttime', 'duration', 'purpose', 'lon', 'lat', 'zone'])
        act_processed = pd.read_csv(processed_file_path, names=['pid', 'age', 'gender', 'occupation', 'starttime', 'duration', 'purpose', 'lon', 'lat', 'zone'])

        last_rows = act_prev.groupby('pid').tail(1)
        last_rows['total_time'] = last_rows['starttime'] + last_rows['duration']

        # Filter rows where total_time != 86400
        incomplete_days = last_rows[last_rows['total_time'] != 86400]

        # Replace corresponding pid data in act_prev
        for _, row in incomplete_days.iterrows():
            pid = row['pid']
            matching_data = act_processed[act_processed['pid'] == pid]
            
            # Remove related data from act_prev and append the replacement data
            act_prev = act_prev[act_prev['pid'] != pid]
            act_prev = pd.concat([act_prev, matching_data], ignore_index=True)

        output_file_path = os.path.join(output_dir, file_name)
        act_prev.to_csv(output_file_path, index=False, header=False)
        print(f"File {file_name} processed and saved to {output_file_path}")

print("All files have been processed.")

## Missing people from household

In [ ]:
pop = pd.read_csv('./data/PseudoPFLOW/ver2.0/agent/8/person_08201.csv', names=['household_id', 'household_type', 'city_code', 'lon', 'lat', 'pid', 'age', 'gender', 'occupation'])
act = pd.read_csv("./data/PseudoPFLOW/ver2.0/activity/8/updated/activity_08201.csv", names=['pid', 'age', 'gender', 'occupation', 'starttime', 'duration', 'purpose', 'lon','lat', 'zone'])
pop_missing = pop[~pop['pid'].isin(act['pid'])]
pop_missing

In [ ]:
pop_missing['occupation'].value_counts()

In [ ]:
pop['occupation'].value_counts()

In [ ]:
len(pop['pid'].unique()), len(act['pid'].unique())

In [ ]:
act